# From CSV to Model: The Workflow

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/T31SHA/data-ml-handbook/blob/main/notebooks/02_workflow.ipynb)

Runnable examples from **Data/ML Handbook** — section `02_workflow`. Read the explanations in the [docs](https://t31sha.github.io/data-ml-handbook/handbook/02_workflow/).

In [ ]:
# Setup: works in Colab or locally from the repo root
import os, subprocess, sys
if not os.path.exists("code") and not os.path.exists("../code"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/T31SHA/data-ml-handbook.git"], check=True)
    os.chdir("data-ml-handbook")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if os.path.exists("../code"): os.chdir("..")
os.chdir("code"); os.makedirs("data", exist_ok=True)
for prep in ["proj01_make_csv.py", "s02_load.py"]:          # shared datasets used by many snippets
    if not os.path.exists("data/housing.parquet"): subprocess.run([sys.executable, prep], check=True)
sys.path.insert(0, "../figures/src")

### `code/s02_load.py`

In [ ]:
import pandas as pd, numpy as np, os
p = "data/housing_raw.csv"
raw = pd.read_csv(p)
typed = pd.read_csv(p, dtype={"region": "category", "HouseAge": "float32", "Population": "float32"},
                    engine="pyarrow")
mb = lambda d: d.memory_usage(deep=True).sum() / 1e6
print(f"default dtypes: {mb(raw):.2f} MB | tuned dtypes: {mb(typed):.2f} MB")
total = sum(chunk.loc[chunk.MedHouseVal > 3, "Population"].sum() for chunk in pd.read_csv(p, chunksize=5_000))
print(f"chunked aggregate over {os.path.getsize(p)/1e6:.1f} MB file: {total:,.0f}")
raw.to_parquet("data/housing.parquet", index=False)
print(f"CSV {os.path.getsize(p)/1e6:.2f} MB -> Parquet {os.path.getsize('data/housing.parquet')/1e6:.2f} MB")
print(pd.read_parquet("data/housing.parquet", columns=["MedInc", "region"]).dtypes.to_dict())

### `code/s02_leakage.py`

In [ ]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
rng = np.random.default_rng(0)
X = rng.normal(size=(200, 5000)); y = rng.integers(0, 2, 200)        # pure noise: true accuracy = 50%
X_leaky = SelectKBest(f_classif, k=20).fit_transform(X, y)              # WRONG: selection sees all labels
print("leaky CV accuracy :", cross_val_score(LogisticRegression(), X_leaky, y, cv=5).mean().round(3))
pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())  # RIGHT: selection inside each fold
print("honest CV accuracy:", cross_val_score(pipe, X, y, cv=5).mean().round(3))